# 03 — Pricing model & relative-value scoring

**Goal:** bring together everything built so far — cleaning, the SQLite database, feature engineering, the hedonic regression model, the comparable-credit model, and relative-value scoring — into one end-to-end run.

**Requires notebook 02 to have been run first** with a real Carbonmark API key, so `data/interim/carbonmark_prices_latest.csv` exists. If you haven't done that yet, this notebook will tell you clearly and stop, rather than failing with a confusing error partway through.

This notebook mostly *calls* functions from `src/` rather than redefining logic inline — that's deliberate. The actual modelling code lives in `src/` where it's testable (see `tests/test_clean.py`) and reusable; this notebook is the narrative that ties it together and shows the results.

### A note on working directory

Colab's default working directory is `/content`, not this notebook's own folder — so relative paths like `data/raw` or `src` only work if the whole repo (not just this one file) is uploaded to `/content` with its folder structure intact, and `/content` is where you're running from.

The cell below finds the repo root automatically (by looking for `requirements.txt`, a file that only exists at the repo root) and changes into it, so the rest of this notebook works regardless of exactly how you got the repo into Colab (zip upload, git clone, or Drive sync). If it can't find the repo root, it tells you clearly instead of failing on some unrelated import error five cells later — this exact class of bug (assuming a working directory that isn't guaranteed) was caught by actually testing this notebook end to end before writing this note.

In [ ]:
import os

def find_and_enter_repo_root(marker_file="requirements.txt", max_up=5):
    """Walk upward from the current directory looking for the repo root
    (identified by the presence of marker_file), and os.chdir into it."""
    current = os.getcwd()
    for _ in range(max_up):
        if os.path.exists(os.path.join(current, marker_file)):
            os.chdir(current)
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    return None

repo_root = find_and_enter_repo_root()
if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repo root (looking for requirements.txt) starting from "
        f"{os.getcwd()}. Make sure the full carbon-credit-quant/ folder (not just this "
        "notebook file) is uploaded to Colab, then set this notebook's runtime working "
        "directory inside it before running further cells."
    )
print(f"Working directory set to repo root: {repo_root}")

In [ ]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.clean import clean_projects, clean_credit_transactions, compute_project_aggregates
from database.db import create_database, load_registries, load_projects, load_credit_transactions
from features.build_features import build_modelling_dataset, get_feature_columns
from models.hedonic import fit_hedonic_model, diagnose_model, predict_hedonic_price
from models.comparable import build_comparable_model, score_comparable_prices
from analysis.relative_value import combine_scores, build_ranked_table
from visualization.plots import (
    plot_price_distribution, plot_actual_vs_predicted,
    plot_residual_distribution, plot_relative_value_ranking,
)

RAW_DIR = "data/raw"
INTERIM_DIR = "data/interim"


## Step 1 — Check prerequisites

Fail loudly and clearly here if notebook 02 hasn't been run, rather than producing a confusing error five cells from now.

In [ ]:
prices_path = f"{INTERIM_DIR}/carbonmark_prices_latest.csv"

if not os.path.exists(prices_path):
    raise FileNotFoundError(
        f"{prices_path} not found. Run notebook 02_carbonmark_prices.ipynb first — "
        f"it needs a real Carbonmark sandbox API key. See that notebook for setup instructions."
    )

prices_raw = pd.read_csv(prices_path)
print(f"Loaded {len(prices_raw)} price observations from {prices_path}")

if len(prices_raw) < 30:
    print(
        f"\n⚠️  Only {len(prices_raw)} price observations available. The hedonic model "
        f"needs at least ~30 complete rows to fit reliably (fewer than that, and the "
        f"regression becomes unstable or fails outright — see src/models/hedonic.py). "
        f"If this is too few, that's real information: it means Carbonmark's coverage "
        f"of OffsetsDB projects is limited, not a bug in this notebook."
    )

## Step 2 — Load and clean OffsetsDB data

Same cleaning functions used in notebook 01, called from `src/data/clean.py` this time instead of redefined inline — one implementation, used everywhere.

In [ ]:
projects_raw = pd.read_csv(f"{RAW_DIR}/projects.csv")
credits_raw = pd.read_csv(f"{RAW_DIR}/credits.csv", low_memory=False)

projects_clean = clean_projects(projects_raw)
credits_clean = clean_credit_transactions(credits_raw)
project_aggregates = compute_project_aggregates(credits_clean)

print(f"Cleaned projects: {len(projects_clean):,}")
print(f"Cleaned credit transactions: {len(credits_clean):,}")
print(f"Project aggregates (issued/retired/remaining): {len(project_aggregates):,}")

## Step 3 — Load into SQLite

This is the actual database step from Phase 4/5 of the project spec — not just pandas in memory, a real queryable database on disk.

In [ ]:
DB_PATH = "carbon_credit_quant.db"
conn = create_database(DB_PATH)

load_registries(conn, projects_clean["registry"].unique().tolist())
load_projects(conn, projects_clean)
load_credit_transactions(conn, credits_clean)

# Confirm with a real SQL query, not just trusting the load functions silently worked
cur = conn.execute("SELECT COUNT(*) FROM projects")
print(f"projects table: {cur.fetchone()[0]:,} rows")
cur = conn.execute("SELECT COUNT(*) FROM credit_transactions")
print(f"credit_transactions table: {cur.fetchone()[0]:,} rows")

## Step 4 — Build the modelling dataset

Joins projects + credit aggregates + Carbonmark prices, and derives the features both models use (project age, liquidity proxy, category supply).

In [ ]:
feature_cols = get_feature_columns()
model_df = build_modelling_dataset(projects_clean, project_aggregates, prices_raw)

print(f"Modelling dataset: {model_df.shape[0]} projects x {model_df.shape[1]} columns")
print(f"\nFeature columns used:")
print(feature_cols)

### A note on `country` before fitting

Testing this pipeline against real project data (with a synthetic price sample, before real Carbonmark access) surfaced a real problem: `country` has enough distinct values that with a small price sample, the regression's dummy-variable matrix becomes rank-deficient (statsmodels will warn `SingularMatrixWarning` and the condition number will be extreme). If you see that warning below, it means the dataset is too small relative to the number of countries — the fix is to either drop `country` from the model, or bucket it into broader regions (e.g. continent) rather than using raw country names. Decide based on what you actually see in the diagnostics output, don't pre-emptively guess.

In [ ]:
result, fitted_df = fit_hedonic_model(model_df, feature_cols)
diagnostics = diagnose_model(result, fitted_df, feature_cols)

print("Model summary:")
print(result.summary())
print()
print("Diagnostics:", diagnostics)

## Step 5 — Fit the comparable-credit model

In [ ]:
nn, feat_matrix, scaler, comparable_pids = build_comparable_model(model_df, feature_cols, k=5)
price_lookup = model_df.set_index("project_id")["price_usd"]
comparable_scores = score_comparable_prices(
    nn, feat_matrix, comparable_pids.reset_index(drop=True),
    price_lookup.loc[comparable_pids].reset_index(drop=True), k=5,
)

print(f"Comparable model scored {len(comparable_scores)} projects")
comparable_scores.head()

## Step 6 — Relative-value scoring

Combines both models' residuals into one score and classification, per `src/analysis/relative_value.py`. Weighting logic and its justification are documented there and in `docs/research_decisions_log.md` — worth reading before trusting the output.

In [ ]:
fitted_df = fitted_df.copy()
fitted_df["hedonic_model_price"] = predict_hedonic_price(result, fitted_df)
fitted_df["observed_price"] = np.exp(fitted_df["log_price"])
fitted_df["project_id"] = model_df.loc[fitted_df.index, "project_id"].values

hedonic_out = fitted_df[["project_id", "observed_price", "hedonic_model_price"]].merge(
    model_df[["project_id", "liquidity_proxy"]], on="project_id", how="left"
)

scored = combine_scores(hedonic_out, comparable_scores)
ranked = build_ranked_table(scored)

print(f"Scored {len(ranked)} projects")
ranked.head(10)

## Step 7 — Visualizations

In [ ]:
fig1 = plot_price_distribution(model_df, price_col="price_usd", by="registry")
plt.show()

In [ ]:
fig2 = plot_actual_vs_predicted(fitted_df["observed_price"], fitted_df["hedonic_model_price"])
plt.show()

In [ ]:
fig3 = plot_residual_distribution(result.resid, title="Hedonic model residual distribution")
plt.show()

In [ ]:
fig4 = plot_relative_value_ranking(ranked, n=15)
plt.show()

## Step 8 — Save results

Writes the ranked table to `data/processed` and a timestamped run into the SQLite `relative_value_scores` table, per the schema — so re-running this notebook builds a history of scoring runs rather than overwriting the past.

In [ ]:
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

run_timestamp = pd.Timestamp.now().isoformat()
ranked_out = ranked.copy()
ranked_out["run_at"] = run_timestamp
ranked_out.to_csv(f"{PROCESSED_DIR}/relative_value_scores_latest.csv", index=False)

cur = conn.cursor()
for _, row in ranked_out.iterrows():
    cur.execute(
        """INSERT INTO relative_value_scores
           (project_id, run_at, observed_price, hedonic_model_price, comparable_model_price,
            residual, residual_zscore, classification)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            row["project_id"], run_timestamp, row["observed_price"],
            row["hedonic_model_price"], row["comparable_model_price"],
            row["observed_price"] - row["hedonic_model_price"], row["combined_zscore"],
            row["classification"],
        ),
    )
conn.commit()
conn.close()

print(f"Saved {len(ranked_out)} scored projects to:")
print(f"  {PROCESSED_DIR}/relative_value_scores_latest.csv")
print(f"  {DB_PATH} (relative_value_scores table, run_at = {run_timestamp})")

## Step 9 — Rebuild the website

Regenerates `reports/website/index.html` with this run's real data embedded. This is what the scheduled GitHub Actions workflow (`.github/workflows/refresh_prices.yml`) commits back to the repo each time it runs — the dashboard should always reflect the most recent scoring run, not a stale snapshot.

In [ ]:
from visualization.build_website import build_dashboard_html

os.makedirs("reports/website", exist_ok=True)
build_dashboard_html(
    ranked_df=ranked,
    diagnostics=diagnostics,
    model_df=model_df,
    fitted_df=fitted_df,
    result_resid=result.resid,
    run_timestamp=run_timestamp,
    output_path="reports/website/index.html",
)
print("Website rebuilt at reports/website/index.html")

## What this notebook does NOT do yet

Being explicit about scope rather than implying this is more complete than it is:

- **No backtest.** There's no historical price series yet (see `docs/research_decisions_log.md`, Decision 4). Re-run notebook 02 periodically to start building one.
- **No environmental-quality score.** Deliberately not fabricated — see the docstring in `src/features/build_features.py`.
- **No full validation suite** (cross-validation, alternative specifications, sensitivity analysis on k for the comparable model). Worth doing once there's enough real price data that the results are worth validating carefully — right now, with a small Carbonmark sample, the priority is getting a defensible baseline running, not over-engineering validation on a small sample.
- **classification labels are "potential" mispricing, not trading signals.** Do not treat a "strong potential undervaluation" tag as an executable arbitrage — see the explicit reasoning in `src/analysis/relative_value.py`.